# 지역별 비교 분석 노트북

시군구별 아파트 매매 데이터를 비교 분석한다.

1. CSV 로드 및 전처리
2. 시군구별 평균 매매가 비교
3. 시군구별 박스플롯 (가격 분포)
4. 시군구 x 월별 거래량 히트맵
5. 평당가 계산 및 비교

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

%matplotlib inline

In [ ]:
# CSV 로드
FILE_PATH = "../data/sample_transactions.csv"

try:
    df = pd.read_csv(FILE_PATH)
    print("데이터 크기:", df.shape)
    print("컬럼:", list(df.columns))
    df.head()
except FileNotFoundError:
    print(f"파일을 찾을 수 없습니다: {FILE_PATH}")
    raise

In [ ]:
# 전처리
# 거래금액 숫자 변환
df['거래금액'] = (
    df['거래금액']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.strip()
    .astype(int)
)

# 전용면적 숫자 변환
df['전용면적'] = pd.to_numeric(df['전용면적'], errors='coerce')

# 날짜 파생 컬럼
df['계약년'] = df['계약년월'].astype(str).str[:4]
df['계약월'] = df['계약년월'].astype(str).str[4:6]
df['월'] = df['계약년'] + '-' + df['계약월']

print("전처리 완료")
df.info()

In [ ]:
# 시군구별 평균 매매가 비교 (막대 차트)
gu_avg = (
    df.groupby('시군구')['거래금액']
    .mean()
    .sort_values(ascending=True)
)

fig, ax = plt.subplots(figsize=(10, max(6, len(gu_avg) * 0.5)))
bars = ax.barh(gu_avg.index, gu_avg.values, color='#3498db', edgecolor='white')

# 값 표시
for bar, val in zip(bars, gu_avg.values):
    ax.text(bar.get_width() + 50, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}만원', ha='left', va='center', fontsize=9)

ax.set_title('시군구별 평균 매매가', fontsize=14, fontweight='bold')
ax.set_xlabel('평균 거래금액 (만원)')
plt.tight_layout()
plt.show()

In [ ]:
# 시군구별 매매가 분포 박스플롯
# 거래 건수가 많은 시군구 상위 10개만 표시
top_gu = df['시군구'].value_counts().head(10).index.tolist()
df_top = df[df['시군구'].isin(top_gu)].copy()

# 시군구별 중앙값 기준으로 정렬
gu_order = (
    df_top.groupby('시군구')['거래금액']
    .median()
    .sort_values(ascending=False)
    .index
)

fig, ax = plt.subplots(figsize=(14, 6))
data_for_box = [df_top[df_top['시군구'] == gu]['거래금액'].values for gu in gu_order]
bp = ax.boxplot(data_for_box, labels=gu_order, patch_artist=True, notch=True)

# 색상 적용
colors_box = plt.cm.Blues(np.linspace(0.3, 0.9, len(gu_order)))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)

ax.set_title('시군구별 매매가 분포 (거래 건수 TOP 10)', fontsize=14, fontweight='bold')
ax.set_ylabel('거래금액 (만원)')
ax.set_xlabel('시군구')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# 시군구 x 월별 거래량 히트맵
volume = df.groupby(['시군구', '월']).size().unstack(fill_value=0)
volume = volume.sort_index(axis=1)  # 월 정렬

fig, ax = plt.subplots(figsize=(14, max(6, len(volume) * 0.4)))
im = ax.imshow(volume.values, aspect='auto', cmap='YlOrRd')

# 축 레이블 설정
ax.set_xticks(range(len(volume.columns)))
ax.set_xticklabels(volume.columns, rotation=45, ha='right', fontsize=8)
ax.set_yticks(range(len(volume.index)))
ax.set_yticklabels(volume.index, fontsize=9)

# 각 셀에 거래 건수 표시 (셀 수가 적을 때만)
if volume.shape[0] * volume.shape[1] <= 200:
    for i in range(volume.shape[0]):
        for j in range(volume.shape[1]):
            val = volume.values[i, j]
            if val > 0:
                text_color = 'white' if val > volume.values.max() * 0.6 else 'black'
                ax.text(j, i, str(val), ha='center', va='center',
                        fontsize=7, color=text_color)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('거래 건수')
ax.set_title('시군구 x 월별 거래량', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# 평당가 계산 (1평 = 3.3058 제곱미터)
PYEONG_FACTOR = 3.3058

df['평수'] = (df['전용면적'] / PYEONG_FACTOR).round(1)
df['평당가'] = (df['거래금액'] / df['평수']).round(0)

# 시군구별 평균 평당가
gu_pyeong = (
    df.groupby('시군구')['평당가']
    .mean()
    .sort_values(ascending=True)
    .round(0)
)

fig, ax = plt.subplots(figsize=(10, max(6, len(gu_pyeong) * 0.5)))
bars = ax.barh(gu_pyeong.index, gu_pyeong.values, color='#e67e22', edgecolor='white')

for bar, val in zip(bars, gu_pyeong.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            f'{val:,.0f}만원/평', ha='left', va='center', fontsize=9)

ax.set_title('시군구별 평균 평당가', fontsize=14, fontweight='bold')
ax.set_xlabel('평당가 (만원/평)')
plt.tight_layout()
plt.show()

In [ ]:
# 시군구별 요약 테이블
summary = df.groupby('시군구').agg(
    거래건수=('거래금액', 'count'),
    평균매매가=('거래금액', 'mean'),
    중앙값=('거래금액', 'median'),
    최저가=('거래금액', 'min'),
    최고가=('거래금액', 'max'),
    평균평당가=('평당가', 'mean'),
    평균전용면적=('전용면적', 'mean')
).round(0)

summary = summary.sort_values('평균매매가', ascending=False)

print("시군구별 요약 통계")
summary

## 분석 요약

- 시군구별 평균 매매가와 평당가를 비교하여 지역 간 가격 차이를 확인할 수 있다.
- 박스플롯을 통해 각 지역 내 가격 편차(이상치 포함)를 파악할 수 있다.
- 월별 거래량 히트맵으로 계절성과 지역별 거래 활성도를 비교할 수 있다.
- 평당가는 면적 차이를 보정한 비교 지표로 활용된다.